# Task 2 - Generate product descriptions

In this notebook, I generate a persuasive 50–90 word product description for every product in the dataset, using a selected model from Nebius Token Factory. For each API call, I collect:
- generated_description
- latency_ms
- input_tokens
- output_tokens

The results are stored in a DataFrame and exported to `assignment_01.xlsx`.

In [15]:
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm
from openai import OpenAI

In [16]:
BASE_DIR = Path.cwd().parent  # notebook is inside /notebooks
DATA_PATH = BASE_DIR / "data" / "products.csv"
OUTPUT_PATH = BASE_DIR / "outputs" / "assignment_01.xlsx"

load_dotenv(BASE_DIR / ".env")

NEBIUS_API_KEY = os.getenv("NEBIUS_API_KEY")
if not NEBIUS_API_KEY:
    raise ValueError("Missing NEBIUS_API_KEY in environment variables.")

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=NEBIUS_API_KEY,
)

In [17]:
df = pd.read_csv(DATA_PATH)
df.head()

,product_name,Product_attribute_list,material,warranty
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty


In [18]:
print(df.columns.tolist())
print(df.shape)

['product_name', 'Product_attribute_list', 'material', 'warranty']
(50, 4)


In [23]:
GENERATION_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct-fast"

In [35]:
SYSTEM_PROMPT = """
You are an expert e-commerce copywriter.

Write exactly one persuasive product description based only on the provided product data.

Requirements:
- Write in a friendly, credible, sales-oriented tone.
- The description must be between 50 and 90 words.
- Use clear, natural, fluent English.
- Do not invent facts, features, materials, benefits, or warranty details that are not explicitly provided.
- Do not exaggerate or add unsupported claims such as "ultimate", "best", or "cutting-edge" unless explicitly supported by the input.
- Focus on the provided attributes, material, and warranty.
- Output only the final product description as a single paragraph.
""".strip()

In [31]:
def build_user_prompt(row: pd.Series) -> str:
    parts = []
    
    for col in df.columns:
        value = row[col]
        if pd.notna(value):
            parts.append(f"{col}: {value}")
    
    product_info = "\n".join(parts)
    
    return f"""Write a product description based on the following product information:

{product_info}
""".strip()

In [32]:
def count_words(text: str) -> int:
    if not text:
        return 0
    return len(text.split())

In [33]:
def generate_description(user_prompt: str, model: str = GENERATION_MODEL) -> dict:
    start = time.perf_counter()

    response = client.chat.completions.create(
        model=model,
        temperature=0.7,
        max_tokens=180,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
    )

    end = time.perf_counter()

    generated_description = response.choices[0].message.content.strip()
    usage = response.usage

    return {
        "generated_description": generated_description,
        "latency_ms": round((end - start) * 1000, 2),
        "input_tokens": usage.prompt_tokens if usage else None,
        "output_tokens": usage.completion_tokens if usage else None,
        "word_count": count_words(generated_description),
    }

In [34]:
sample_row = df.iloc[0]
sample_prompt = build_user_prompt(sample_row)

print(sample_prompt)
result = generate_description(sample_prompt)
result

Write a product description based on the following product information:

product_name: Apple iPhone 15 Pro
Product_attribute_list: features: A17 Pro chip, 120 Hz ProMotion display, USB‑C fast charging; dimensions: compact
material: titanium frame, Ceramic Shield glass
warranty: 1‑year limited warranty


{'generated_description': "Experience the future of mobile technology with the Apple iPhone 15 Pro. Powered by the lightning-fast A17 Pro chip, this compact device delivers seamless performance and a stunning 120Hz ProMotion display. The durable titanium frame and Ceramic Shield glass ensure a premium feel that can withstand the rigors of daily life. Fast charge your iPhone 15 Pro via USB-C and enjoy a year of worry-free use, backed by Apple's 1-year limited warranty.",
 'latency_ms': 1225.66,
 'input_tokens': 198,
 'output_tokens': 92,
 'word_count': 72}

In [37]:
results = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    user_prompt = build_user_prompt(row)

    try:
        result = generate_description(user_prompt)
    except Exception as e:
        result = {
            "generated_description": None,
            "latency_ms": None,
            "input_tokens": None,
            "output_tokens": None,
            "word_count": None,
            "error": str(e),
        }

    combined = row.to_dict()
    combined["user_prompt"] = user_prompt
    combined.update(result)
    results.append(combined)

100%|██████████| 50/50 [00:38<00:00,  1.31it/s]


In [38]:
results_df = pd.DataFrame(results)

rubric_columns = [
    "fluency",
    "grammar",
    "tone",
    "length",
    "grounding",
    "latency_rating",
    "cost_rating",
    "final_score",
]

for col in rubric_columns:
    results_df[col] = ""

if "error" not in results_df.columns:
    results_df["error"] = ""

results_df.head()

,product_name,Product_attribute_list,material,warranty,user_prompt,generated_description,latency_ms,input_tokens,output_tokens,word_count,fluency,grammar,tone,length,grounding,latency_rating,cost_rating,final_score,error
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,Write a product description based on the follo...,"""Get ready to experience the latest and greate...",1085.23,216,88,70,,,,,,,,,
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Write a product description based on the follo...,"""Unlock unparalleled mobile photography with t...",604.66,219,90,66,,,,,,,,,
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Write a product description based on the follo...,"""Take your mobile experience to the next level...",704.38,217,115,87,,,,,,,,,
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,Write a product description based on the follo...,"Immerse yourself in rich, immersive sound with...",700.14,216,113,80,,,,,,,,,
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,Write a product description based on the follo...,"""Experience unparalleled sound and comfort wit...",630.15,210,85,60,,,,,,,,,


In [39]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
results_df.to_excel(OUTPUT_PATH, index=False)

print(f"Saved results to: {OUTPUT_PATH}")

Saved results to: /Users/mosheoz/Desktop/repos/llm-evaluation-pipeline/outputs/assignment_01.xlsx
